In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
import torchvision.models as models
from sklearn.model_selection import train_test_split
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np
import warnings
import re

warnings.filterwarnings('ignore') # Pour cacher les faux avertissements de PyTorch

video_dir = "../sia-predicting-short-form-video-popularity/videos_mp4/downloads"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATASET ---
class TikTokDataset(Dataset):
    def __init__(self, video_df, audio_df, label_df):
        # Scan ultra rapide
        self.video_map = {}
        for p in Path(video_dir).rglob('*.mp4'):
            digits = re.findall(r'\d+', p.name)
            if digits:
                self.video_map[digits[0]] = str(p)
        
        # Nettoyage et Alignement parfaits
        video_df = video_df.copy()
        audio_df = audio_df.copy()
        label_df = label_df.copy()
        
        video_df['id'] = video_df['id'].astype(str).str.strip()
        audio_df['id'] = audio_df['id'].astype(str).str.strip()
        id_col_label = 'ID' if 'ID' in label_df.columns else 'id'
        label_df[id_col_label] = label_df[id_col_label].astype(str).str.strip()

        common_ids = sorted(list(set(video_df['id']) & set(audio_df['id']) & set(label_df[id_col_label]) & set(self.video_map.keys())))
        
        self.video_data = video_df[video_df['id'].isin(common_ids)].sort_values('id').reset_index(drop=True)
        self.audio_features = audio_df[audio_df['id'].isin(common_ids)].sort_values('id').reset_index(drop=True)
        self.label_data = label_df[label_df[id_col_label].isin(common_ids)].sort_values(id_col_label).reset_index(drop=True)
        
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.y_train = self.label_data['popularity']
        print(f"Dataset prêt et aligné : {len(self.video_data)} vidéos.")

    def __len__(self):
        return len(self.video_data)

    def __getitem__(self, idx):
        try:
            video_id = str(self.video_data.iloc[idx]['id'])
            video_path = self.video_map[video_id]
            
            # --- LECTURE SÉQUENTIELLE INTELLIGENTE ---
            cap = cv2.VideoCapture(video_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            if total_frames <= 0:
                cap.release()
                raise ValueError("Vidéo illisible")

            # On lit une frame sur X pour avoir 30 candidates rapides sans saturer la RAM
            skip_rate = max(1, total_frames // 30)
            candidate_frames = []
            motion_scores = []
            prev_gray = None
            count = 0
            
            # Lecture fluide (pas de "cap.set" qui fait planter les MP4 compressés !)
            while True:
                ret, frame = cap.read()
                if not ret: break # Fin de la vidéo
                
                if count % skip_rate == 0:
                    # Redimensionnement immédiat pour économiser la mémoire
                    frame_small = cv2.resize(frame, (224, 224))
                    
                    # Passage en gris pour calculer l'action
                    gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)
                    
                    # Calcul du score d'action (différence avec la frame précédente)
                    if prev_gray is None:
                        score = 0
                    else:
                        score = np.mean(cv2.absdiff(gray, prev_gray))
                    
                    prev_gray = gray
                    
                    # On sauvegarde la frame propre (en RGB) et son score
                    frame_rgb = cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB)
                    candidate_frames.append(frame_rgb)
                    motion_scores.append(score)
                    
                    if len(candidate_frames) >= 30: break # On a assez de candidates
                
                count += 1
            
            cap.release()

            # --- SÉLECTION DES FRAMES LES PLUS PERTINENTES ---
            if len(candidate_frames) < 8:
                # Si la vidéo est vraiment très courte, on la remplit en dupliquant la dernière image
                while len(candidate_frames) < 8:
                    candidate_frames.append(candidate_frames[-1] if candidate_frames else np.zeros((224,224,3), dtype=np.uint8))
                selected_frames = candidate_frames[:8]
            else:
                # On prend les index des 8 frames avec LE PLUS D'ACTION
                top_indices = np.argsort(motion_scores)[-8:]
                # On les trie chronologiquement pour que le modèle GRU comprenne l'histoire
                top_indices = sorted(top_indices)
                selected_frames = [candidate_frames[i] for i in top_indices]

            # Conversion Tenseur [8, 3, 224, 224]
            vframes = torch.from_numpy(np.stack(selected_frames)).permute(0, 3, 1, 2).float() / 255.0
            
            # --- TEXTE ---
            desc = str(self.video_data.iloc[idx]['description'])
            text_in = self.tokenizer(desc, padding='max_length', max_length=50, truncation=True, return_tensors="pt")

            # --- TAB & AUDIO ---
            tab_data = torch.tensor(self.video_data.iloc[idx][['video_duration', 'aspect_ratio']].values.astype(float), dtype=torch.float)
            audio_data = torch.tensor(self.audio_features.iloc[idx, 1:].values.astype(float), dtype=torch.float)
            label = torch.tensor(float(self.y_train.iloc[idx]), dtype=torch.float)

            return vframes, text_in['input_ids'].squeeze(), tab_data, audio_data, label

        except Exception as e:
            # En cas d'erreur fatale, on skip
            return self.__getitem__((idx + 1) % len(self))

# --- 2. MODÈLE ---
class TikTokModel(nn.Module):
    def __init__(self, num_tab_features=2):
        super(TikTokModel, self).__init__()
        resnet = models.resnet18(weights='DEFAULT')
        self.video_backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.video_gru = nn.GRU(input_size=512, hidden_size=256, batch_first=True)
        
        self.text_bert = BertModel.from_pretrained('bert-base-uncased', use_safetensors=False)
        for param in self.text_bert.parameters(): param.requires_grad = False
        
        self.tabular = nn.Sequential(nn.Linear(num_tab_features, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU())
        self.audio = nn.Sequential(nn.Linear(34, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 64), nn.ReLU())
        
        self.regressor = nn.Sequential(nn.Linear(256 + 768 + 16 + 64, 128), nn.ReLU(), nn.Linear(128, 1))
        
    def forward(self, x_video, x_text, x_tab, x_audio):
        batch_size = x_video.size(0)
        
        # Vidéo
        x_v = self.video_backbone(x_video.view(-1, 3, 224, 224)).view(batch_size, 8, 512)
        _, hidden = self.video_gru(x_v)
        video_features = hidden.squeeze(0)
        
        # Texte (Corrigé: plus de x_mask)
        text_features = self.text_bert(x_text).pooler_output
        
        # Tab et Audio
        tab_features = self.tabular(x_tab)
        audio_features = self.audio(x_audio)

        combined = torch.cat((video_features, text_features, tab_features, audio_features), dim=1)
        return self.regressor(combined)

# --- 3. PRÉPARATION DES DONNÉES ---
video_df = pd.read_csv('X_train.csv', sep=';')
audio_df = pd.read_csv('audio_features_ready.csv', sep=',')
label_df = pd.read_csv('y_train.csv', sep=';')

for df in [video_df, audio_df, label_df]:
    df.columns = df.columns.str.strip()
    id_col = 'id' if 'id' in df.columns else 'ID'
    df[id_col] = df[id_col].astype(str).str.strip()

df_train, df_val = train_test_split(video_df, test_size=0.2, random_state=42)
audio_train = audio_df[audio_df['id'].isin(df_train['id'])]
audio_val = audio_df[audio_df['id'].isin(df_val['id'])]
label_train = label_df[label_df['ID'].isin(df_train['id'])]
label_val = label_df[label_df['ID'].isin(df_val['id'])]

train_dataset = TikTokDataset(video_df=df_train, audio_df=audio_train, label_df=label_train)
val_dataset = TikTokDataset(video_df=df_val, audio_df=audio_val, label_df=label_val)

# On peut remettre un batch_size correct (4) car OpenCV est très léger
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

model = TikTokModel().to(device)
criterion = nn.MSELoss()  
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# --- 4. ENTRAÎNEMENT ---
epochs = 1
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for batch in pbar:
        # 5 éléments récupérés, tout s'aligne !
        frames, text, tab, audio, labels = batch
        frames, text, tab, audio, labels = frames.to(device), text.to(device), tab.to(device), audio.to(device), labels.to(device)
        
        outputs = model(frames, text, tab, audio)
        loss = criterion(outputs.squeeze(), labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    print(f"Fin de l'Epoch {epoch+1} - Loss moyenne: {train_loss/len(train_loader):.4f}")

# --- 5. INFÉRENCE SUR LE TEST SET ET CRÉATION DU CSV ---
print("\n🚀 Préparation des prédictions pour la soumission...")

# 1. Charger les données de test
df_test = pd.read_csv('X_test.csv', sep=';') # Attention au sep=';'
audio_test_df = pd.read_csv('audio_features_ready.csv', sep=',') 

# Nettoyage des IDs (la règle d'or)
df_test.columns = df_test.columns.str.strip()
df_test['id'] = df_test['id'].astype(str).str.strip()

audio_test_df.columns = audio_test_df.columns.str.strip()
audio_test_df['id'] = audio_test_df['id'].astype(str).str.strip()

# Créer une fausse colonne 'popularity' pour tromper ton TikTokDataset
df_test_labels = df_test.copy()
df_test_labels['popularity'] = 0.0  # Valeur factice

# 2. Création du Dataset et DataLoader
test_dataset = TikTokDataset(video_df=df_test, audio_df=audio_test_df, label_df=df_test_labels)
# Pas de shuffle=True ici, on veut garder l'ordre !
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

# 3. Boucle de prédiction
model.eval() # Très important : désactive le dropout pour des prédictions stables
predictions = []

with torch.no_grad(): # Désactive le calcul des gradients (va beaucoup plus vite)
    pbar_test = tqdm(test_loader, desc="Prédictions en cours")
    for batch in pbar_test:
        frames, text, tab, audio, _ = batch # Le '_' ignore notre label factice
        
        frames, text = frames.to(device), text.to(device)
        tab, audio = tab.to(device), audio.to(device)
        
        # Inférence
        outputs = model(frames, text, tab, audio)
        
        # On récupère les valeurs
        preds = outputs.squeeze().cpu().numpy().tolist()
        
        if isinstance(preds, float):
            predictions.append(preds)
        else:
            predictions.extend(preds)

# 4. Alignement sécurisé des prédictions avec les IDs passés dans le modèle
ids_predits = test_dataset.video_data['id'].tolist()
preds_df = pd.DataFrame({'id': ids_predits, 'popularity': predictions})



# 5. La parade anti-crash : on fusionne avec le VRAI X_test
final_submission = df_test[['id']].copy()
final_submission = final_submission.merge(preds_df, on='id', how='left')

# Si des vidéos ont été ignorées (frames manquantes), on remplace le NaN par la moyenne
mean_pop = final_submission['popularity'].mean()
final_submission['popularity'] = final_submission['popularity'].fillna(mean_pop)

# 6. Sauvegarde
nom_fichier = 'submission_tiktok_finale.csv'
final_submission.to_csv(nom_fichier, index=False)

print(f"\n✅ Terminé ! Fichier '{nom_fichier}' généré.")
print(f"Lignes attendues : {len(df_test)} | Lignes générées : {len(final_submission)}")

Dataset prêt et aligné : 1078 vidéos.
Dataset prêt et aligné : 270 vidéos.


Epoch 1/1 [Train]: 100%|██████████| 270/270 [2:15:26<00:00, 30.10s/it, loss=nan]    


Fin de l'Epoch 1 - Loss moyenne: nan

🚀 Préparation des prédictions pour la soumission...


FileNotFoundError: [Errno 2] No such file or directory: 'X_test.csv'

In [4]:
# --- 5. INFÉRENCE SUR LE TEST SET ET CRÉATION DU CSV ---
print("\n🚀 Préparation des prédictions pour la soumission...")

# 1. Charger les données de test
df_test = pd.read_csv('X_test.csv', sep=';') # Attention au sep=';'
audio_test_df = pd.read_csv('audio_features_ready.csv', sep=',') 

# Nettoyage des IDs (la règle d'or)
df_test.columns = df_test.columns.str.strip()
df_test['id'] = df_test['id'].astype(str).str.strip()

audio_test_df.columns = audio_test_df.columns.str.strip()
audio_test_df['id'] = audio_test_df['id'].astype(str).str.strip()

# Créer une fausse colonne 'popularity' pour tromper ton TikTokDataset
df_test_labels = df_test.copy()
df_test_labels['popularity'] = 0.0  # Valeur factice

# 2. Création du Dataset et DataLoader
test_dataset = TikTokDataset(video_df=df_test, audio_df=audio_test_df, label_df=df_test_labels)
# Pas de shuffle=True ici, on veut garder l'ordre !
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

# 3. Boucle de prédiction
model.eval() # Très important : désactive le dropout pour des prédictions stables
predictions = []

with torch.no_grad(): # Désactive le calcul des gradients (va beaucoup plus vite)
    pbar_test = tqdm(test_loader, desc="Prédictions en cours")
    for batch in pbar_test:
        frames, text, tab, audio, _ = batch # Le '_' ignore notre label factice
        
        frames, text = frames.to(device), text.to(device)
        tab, audio = tab.to(device), audio.to(device)
        
        # Inférence
        outputs = model(frames, text, tab, audio)
        
        # On récupère les valeurs
        preds = outputs.squeeze().cpu().numpy().tolist()
        
        if isinstance(preds, float):
            predictions.append(preds)
        else:
            predictions.extend(preds)

# 4. Alignement sécurisé des prédictions avec les IDs passés dans le modèle
ids_predits = test_dataset.video_data['id'].tolist()
preds_df = pd.DataFrame({'id': ids_predits, 'popularity': predictions})



# 5. La parade anti-crash : on fusionne avec le VRAI X_test
final_submission = df_test[['id']].copy()
final_submission = final_submission.merge(preds_df, on='id', how='left')

# Si des vidéos ont été ignorées (frames manquantes), on remplace le NaN par la moyenne
mean_pop = final_submission['popularity'].mean()
final_submission['popularity'] = final_submission['popularity'].fillna(mean_pop)

# 6. Sauvegarde
nom_fichier = 'submission_tiktok_finale.csv'
final_submission.to_csv(nom_fichier, index=False)

print(f"\n✅ Terminé ! Fichier '{nom_fichier}' généré.")
print(f"Lignes attendues : {len(df_test)} | Lignes générées : {len(final_submission)}")


🚀 Préparation des prédictions pour la soumission...
Dataset prêt et aligné : 338 vidéos.


Prédictions en cours: 100%|██████████| 85/85 [23:26<00:00, 16.55s/it]



✅ Terminé ! Fichier 'submission_tiktok_finale.csv' généré.
Lignes attendues : 338 | Lignes générées : 338
